In [19]:
from pyspark.sql import SparkSession

In [20]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [21]:
spark = SparkSession.builder.appName("case-study-notes").getOrCreate()

In [23]:
customers_df = spark.read.csv("Data/customers.csv", header=True, inferSchema=True)
order_items_df = spark.read.csv("Data/order_items.csv", header=True, inferSchema=True)
orders_df = spark.read.csv("Data/orders.csv", header=True, inferSchema=True)
products_df = spark.read.csv("Data/products.csv", header=True, inferSchema=True)
returns_df = spark.read.csv("Data/returns.csv", header=True, inferSchema=True)

In [24]:
customers_df.createOrReplaceTempView("customers")
order_items_df.createOrReplaceTempView("order_items")
orders_df.createOrReplaceTempView("orders")
products_df.createOrReplaceTempView("products")
returns_df.createOrReplaceTempView("returns")

In [26]:
customers_df.show(5)
order_items_df.show(5)
orders_df.show(5)
products_df.show(5)
returns_df.show(5)

+-----------+-------------+--------+-----+-----------------+----------------+
|customer_id|customer_name|    city|state|registration_date|customer_segment|
+-----------+-------------+--------+-----+-----------------+----------------+
|          1|   Customer_1|Columbus|   OH|       2023-10-17|             VIP|
|          2|   Customer_2|   Miami|   CA|       2022-04-25|         Premium|
|          3|   Customer_3| Atlanta|   FL|       2022-01-26|         Premium|
|          4|   Customer_4| Chicago|   OH|       2022-10-09|        Standard|
|          5|   Customer_5|Columbus|   IL|       2022-09-08|         Premium|
+-----------+-------------+--------+-----+-----------------+----------------+
only showing top 5 rows

+-------------+--------+----------+--------+-------------+
|order_item_id|order_id|product_id|quantity|selling_price|
+-------------+--------+----------+--------+-------------+
|            1|  227444|     28849|       5|       727.98|
|            2|   32708|     25471|  

In [28]:
sql_df1 = spark.sql("""
    SELECT 
        (SELECT count(customer_id) FROM customers) as total_customers,
        (SELECT count(product_id) FROM products) as total_products,
        (SELECT count(order_id) FROM orders) as total_orders,
        (SELECT count(order_item_id) FROM order_items) as total_order_items,
        (SELECT count(return_id) FROM returns) as total_returns
""")

sql_df1.show()

+---------------+--------------+------------+-----------------+-------------+
|total_customers|total_products|total_orders|total_order_items|total_returns|
+---------------+--------------+------------+-----------------+-------------+
|         100000|         50000|     1000000|          3000000|       100000|
+---------------+--------------+------------+-----------------+-------------+



In [39]:
sql_df1.write.mode("overwrite").csv("output/result1", header = True)

In [32]:
sql_df2 = spark.sql("""
    SELECT category as product_category, 
    sum(unit_cost) as total_sales_amount
    from products
    group by category
""")

sql_df2.show()

+----------------+------------------+
|product_category|total_sales_amount|
+----------------+------------------+
|  Home & Kitchen| 2901364.330000004|
|          Sports| 2853163.040000003|
|     Electronics|2864604.7399999946|
|        Clothing| 2841424.610000002|
|           Books|2853871.8500000075|
|          Beauty|2919388.7500000037|
|            Toys|2851913.1100000013|
+----------------+------------------+



In [40]:
sql_df2.write.mode("overwrite").csv("output/result2", header = True)

In [41]:
sql_df3 = spark.sql("""
    SELECT  c.customer_id, 
    c.customer_name as customer,
    round(sum(oi.quantity*oi.selling_price),2) as total_purchase_amount
    FROM
    customers c 
    join orders o
    on c.customer_id = o.customer_id
    join order_items oi
    on o.order_id = oi.order_id
    where o.order_status = 'Delivered'
    group by c.customer_id,customer
    order by total_purchase_amount DESC
    limit 10

""")

sql_df3.show()

[Stage 129:============================>                            (1 + 1) / 2]

+-----------+--------------+---------------------+
|customer_id|      customer|total_purchase_amount|
+-----------+--------------+---------------------+
|      64560|Customer_64560|            119030.04|
|      65135|Customer_65135|            111137.38|
|      52275|Customer_52275|            108098.33|
|      28584|Customer_28584|            107848.24|
|      37277|Customer_37277|            107444.08|
|      17810|Customer_17810|            105355.56|
|      11201|Customer_11201|             104247.4|
|      33876|Customer_33876|             102933.8|
|      10188|Customer_10188|            100520.46|
|      97963|Customer_97963|             99773.07|
+-----------+--------------+---------------------+



In [42]:
sql_df2.write.mode("overwrite").csv("output/result3", header = True)

In [43]:
sql_df4 = spark.sql("""
    WITH LatestYear AS (
        SELECT MAX(YEAR(order_date)) as max_year 
        FROM orders
    )
    
    SELECT 
        MONTH(o.order_date) as sales_month,
        ROUND(SUM(oi.quantity * oi.selling_price), 2) as total_revenue
    FROM orders o
    JOIN order_items oi 
        ON o.order_id = oi.order_id
    JOIN LatestYear ly 
        ON YEAR(o.order_date) = ly.max_year
    WHERE 
        o.order_status = 'Delivered'
    GROUP BY 
        MONTH(o.order_date)
    ORDER BY 
        sales_month ASC
""")

sql_df4.show()

[Stage 152:============================>                            (1 + 1) / 2]

+-----------+--------------+
|sales_month| total_revenue|
+-----------+--------------+
|          1|2.2365795123E8|
|          2| 2.077295194E8|
|          3|2.2339561183E8|
|          4|2.1220716088E8|
|          5|2.2272812783E8|
|          6|2.1422183647E8|
|          7|2.2192965891E8|
|          8|2.2081820508E8|
|          9|2.1698195885E8|
|         10|2.1992173365E8|
|         11|2.1664222997E8|
|         12|2.2202115806E8|
+-----------+--------------+

